<a href="https://colab.research.google.com/github/SmdIoufy/Gestion-Tontine/blob/master/Copie_de_Sentiment_Analysis_Using_SimpleRNN_LSTM_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### <font color='#00d2d3'> Importer les packages

In [13]:
# Importer les packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### <font color='#00d2d3'> Importer et lire les données

In [16]:
# Importer les données
df = pd.read_csv('/content/drive/MyDrive/DL 6/tweet_dataset.csv')
df.head()

,label,tweet,new_tweet
0,1,@user homegrown rightie white americans are 10...,homegrown rightie white american time likely h...
1,1,"@user @user is this a new one? either way, #m...",new one either way men disgraced harassment ve...
2,0,i am thankful for pillows. #thankful #positive...,thankful pillow thankful positive
3,0,it's unfounate today that aids is under contro...,unfounate today aid control hate kill gay year...
4,0,i am super. #i_am #positive #affirmation,super positive affirmation


- 1 : Negative
- 0 : Positive

In [17]:
# Dimension des données
df.shape

(4484, 3)

### <font color='#00d2d3'> Exploration des données

In [18]:
# Nombre d'observations dans chaque classe
val = df.label.value_counts()
val

,count
label,
1,2242
0,2242


In [19]:
# Bar plot des nombres d'observations dans chaque classe
import plotly.express as px
fig = px.bar(x = val.index, y = val.values, color = val.index.astype(str), text_auto = True)
fig.show()

In [20]:
# Afficher la quatrième ligne de la colonne tweet
df['tweet'][4]

'i am super. #i_am #positive #affirmation     '

In [21]:
# Afficher la quatrième ligne  de la colonne new_tweet
df['new_tweet'][4]

'super positive affirmation'

In [ ]:
# tweet1 = 'Je vais à la banque.'----------vectorisation -------Vec1
# tweet2 = 'Je suis malade.' ----------vectorisation -------Vec2
# tweet3 = 'La banque est loin de la maison' --------Tokenisation----- ['La', 'banque', 'est', 'loin', 'de', 'la', 'maison']
# corpus_CC = [Je, vais, à, la, banque, suis, malade, est, loin, de, maison]
# tweet1 = [1, 2, 3, 4, 5]--------- Padding ----[1, 2, 3, 4, 5, 0, 0]
# tweet2 = [1, 6, 7] --------- Padding ----[1, 6, 7, 0, 0, 0, 0]
# tweet3 = [4, 5, 8, 9, 10, 4, 11]

### <font color='#00d2d3'> Vectorisation

In [22]:
# Importer les packages
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [23]:
# Nombre de mots unique
voc_size = len(set(" ".join([str( a) for a in list(df.new_tweet.values)]).split()))
# Initialiser le modèle Tokenizer
tokenizer = Tokenizer(num_words=voc_size, split=' ')
# Entrainer les données
tokenizer.fit_on_texts(df['new_tweet'].values.astype(str))
# Vectoriser
X = tokenizer.texts_to_sequences(df['new_tweet'].values.astype(str))
# Padding
X = pad_sequences(X)
X[:5]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0, 3793, 3794,    6,   99,   14, 1381, 1784,  998,  296],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    9,   16,  999,   47,   85, 1785, 1000, 2402],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,   33, 2403,   33,   39],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0, 3795,   17, 2404,  536,   26,  257,  308, 3796,  111],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,  415,   39,  132]],
      dtype=int32)

In [24]:
# Importer train_test_split
from sklearn.model_selection import train_test_split
# Labels
y = df.label.values
# Splitter en train et test
x_train, x_vt, y_train, y_vt = train_test_split(X,y, test_size =0.2, random_state = 42)
# Splitter validation et test
x_val, x_test, y_val, y_test = train_test_split(x_vt,y_vt, test_size =0.5, random_state = 42)

### <font color='#00d2d3'> Sentiment Analysis

In [25]:
from keras.models import Sequential
from keras.layers import Dense, Embedding, SimpleRNN,LSTM,GRU, Dropout

In [26]:
# Installer keras tuner
!pip install keras-tuner --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-tuner]


In [27]:
# importer le package keras_tuner
import keras_tuner as kt

In [28]:
# tweet1 = [1, 2, 3, 4, 5] ------- dim == 2 -- [[0.1, 0.2], [0.5, 0.9], [0.3, 0.5]]
# tweet2 = [1, 6, 7]
# tweet3 = [4, 5, 8, 9, 10, 4, 11]

In [29]:
def build_model(hp):
  model = Sequential() # initialiser l'architecture

  # Choisir le nombre de neurones des couches RNN
  hp_units1 = hp.Int('units1', min_value = 50, max_value = 200, step = 50)
  hp_units2 = hp.Int('units2', min_value = 50, max_value = 200, step = 50)
  # Choisir la dimension des features embedding optimal pour la couche embedding
  hp_emb_dim = hp.Int('emb_dim', min_value = 20, max_value = 100, step = 20)
  # Simple RNN architecture
  if  model_name == 'SimpleRNN':
      model=Sequential()
      model.add(Embedding(voc_size, hp_emb_dim, input_length=x_train.shape[1])) # Couche Embedding
      model.add(SimpleRNN(hp_units1, return_sequences=True)) # Couche RNN 1
      model.add(Dropout(0.001))
      model.add(SimpleRNN(hp_units2))# Couche RNN 2
      model.add(Dropout(0.001))
      model.add(Dense(1,activation='sigmoid')) # Couche de sortie
      # compiler le modèle
      model.compile(optimizer = 'adam', loss='binary_crossentropy', metrics = ['accuracy'])
  elif model_name =='LSTM':
      model=Sequential()
      model.add(Embedding(voc_size, hp_emb_dim, input_length=x_train.shape[1])) # Couche Embedding
      model.add(LSTM(hp_units1, return_sequences=True)) # Couche LSTM 1
      model.add(Dropout(0.001))
      model.add(LSTM(hp_units2))# Couche LSTM 2
      model.add(Dropout(0.001))
      model.add(Dense(1,activation='sigmoid')) # Couche de sortie
      # compiler le modèle
      model.compile(optimizer = 'adam', loss='binary_crossentropy', metrics = ['accuracy'])
  else:
      model=Sequential()
      model.add(Embedding(voc_size, hp_emb_dim, input_length=x_train.shape[1])) # Couche Embedding
      model.add(GRU(hp_units1, return_sequences=True)) # Couche GRU 1
      model.add(Dropout(0.001))
      model.add(GRU(hp_units2))# Couche GRU 2
      model.add(Dropout(0.001))
      model.add(Dense(1,activation='sigmoid')) # Couche de sortie
      # compiler le modèle
      model.compile(optimizer = 'adam', loss='binary_crossentropy', metrics = ['accuracy'])

  return model


#### <font color='#fff00'> SimpleRNN Model

##### <font color='#00aaaa'> Recherche de paramètres optimaux pour le SimpleRNN

In [30]:
model_name = 'SimpleRNN'
# Initialiser le tuner
tuner_rnn = kt.Hyperband(build_model,
                    objective='accuracy',
                    max_epochs=10,
                    directory='SimpleRNN_dir')

# lancer la recherche avec le tuner
tuner_rnn.search(x_train, y_train, validation_data = (x_val, y_val), epochs = 10)

Trial 30 Complete [00h 00m 18s]
accuracy: 0.999442458152771

Best accuracy So Far: 0.999442458152771
Total elapsed time: 00h 06m 51s


In [31]:
# Afficher les paramètres optimaux
for h_param in [f"units{i}" for i in range(1,3)] + ['emb_dim']:
  print(h_param, tuner_rnn.get_best_hyperparameters()[0].get(h_param))

units1 100
units2 150
emb_dim 100


In [32]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
# Architecture de modèle SimpleRNN avec les paramètres optimaux
rnn_model=Sequential()
rnn_model.add(Embedding(voc_size,80, input_length=x_train.shape[1])) # Couche Embedding
rnn_model.add(SimpleRNN(100, return_sequences=True)) # Couche RNN 1
rnn_model.add(Dropout(0.001))
rnn_model.add(SimpleRNN(100))# Couche RNN 2
rnn_model.add(Dropout(0.001))
rnn_model.add(Dense(1,activation='sigmoid')) # Couche de sortie
# Compiler le modèle
rnn_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [33]:
# Entrainer le modèle
histore_rnn = rnn_model.fit(x_train, y_train, validation_data = (x_val, y_val), epochs = 20, callbacks=early_stopping)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - accuracy: 0.5927 - loss: 0.6403 - val_accuracy: 0.7411 - val_loss: 0.4867
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9554 - loss: 0.1224 - val_accuracy: 0.7924 - val_loss: 0.4626
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9955 - loss: 0.0146 - val_accuracy: 0.8080 - val_loss: 0.5061
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9992 - loss: 0.0053 - val_accuracy: 0.8326 - val_loss: 0.5146
Epoch 5/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9992 - loss: 0.0034 - val_accuracy: 0.8237 - val_loss: 0.5481


In [34]:
# Accuracy et loss sur les données validation
print(f"Accuracy : {rnn_model.evaluate(x_val, y_val)[1]}  ---- Loss : {rnn_model.evaluate(x_val, y_val)[0]}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8237 - loss: 0.5481
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8237 - loss: 0.5481
Accuracy : 0.8236607313156128  ---- Loss : 0.5480509400367737


#### <font color='#fff00'> LSTM Model

##### <font color='#00aaaa'> Recherche de paramètres optimaux pour le LSTM

In [35]:
model_name = 'LSTM'
# Initialiser le tuner
tuner_lstm = kt.Hyperband(build_model,
                    objective='accuracy',
                    max_epochs=10,
                    directory='LSTM_dir')

# lancer la recherche avec le tuner
tuner_lstm.search(x_train, y_train, validation_data = (x_val, y_val), epochs = 10)

Trial 30 Complete [00h 00m 18s]
accuracy: 0.9988848567008972

Best accuracy So Far: 0.999442458152771
Total elapsed time: 00h 04m 30s


In [36]:
# Afficher les paramètres optimaux
for h_param in [f"units{i}" for i in range(1,3)] +['emb_dim']:
  print(h_param, tuner_lstm.get_best_hyperparameters()[0].get(h_param))

units1 100
units2 150
emb_dim 100


In [44]:
# Architecture de modèle LSTM avec les paramètres optimaux
lstm_model=Sequential()
lstm_model.add(Embedding(voc_size,100,input_length=x_train.shape[1])) # Couche Embedding
lstm_model.add(LSTM(100, return_sequences=True)) # Couche LSTM 1
lstm_model.add(Dropout(0.001))
lstm_model.add(LSTM(150))# Couche LSTM 2
lstm_model.add(Dropout(0.001))
lstm_model.add(Dense(1,activation='sigmoid')) # Couche de sortie
# Compiler le modèle
lstm_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [45]:
# Entrainer le modèle
history_lstm = lstm_model.fit(x_train, y_train, validation_data = (x_val, y_val), epochs = 20, callbacks = early_stopping)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7313 - loss: 0.4908 - val_accuracy: 0.8527 - val_loss: 0.3112
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9445 - loss: 0.1459 - val_accuracy: 0.8839 - val_loss: 0.3069
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9866 - loss: 0.0438 - val_accuracy: 0.8772 - val_loss: 0.4228


In [41]:
# Accuracy et loss sur les données validation
print(f"Accuracy : {lstm_model.evaluate(x_val, y_val)[1]}  ---- Loss : {lstm_model.evaluate(x_val, y_val)[0]}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8728 - loss: 0.4533
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8728 - loss: 0.4533 
Accuracy : 0.8727678656578064  ---- Loss : 0.45329612493515015


#### <font color='#fff00'> GRU Model

In [46]:
model_name = 'GRU'
# Initialiser le tuner
tuner_gru = kt.Hyperband(build_model,
                    objective='accuracy',
                    max_epochs=10,
                    directory='GRU_dir')

# lancer la recherche avec le tuner
tuner_gru.search(x_train, y_train, validation_data = (x_val, y_val), epochs = 10)

Reloading Tuner from GRU_dir/untitled_project/tuner0.json


In [47]:
# Afficher les paramètres optimaux
for h_param in [f"units{i}" for i in range(1,3)] +['emb_dim']:
  print(h_param, tuner_gru.get_best_hyperparameters()[0].get(h_param))

units1 100
units2 200
emb_dim 100


In [56]:
# Architecture de modèle LSTM avec les paramètres optimaux
gru_model=Sequential()
gru_model.add(Embedding(voc_size,40,input_length=x_train.shape[1])) # Couche Embedding
gru_model.add(GRU(150, return_sequences=True)) # Couche LSTM 1
gru_model.add(Dropout(0.001))
gru_model.add(GRU(50))# Couche LSTM 2
gru_model.add(Dropout(0.001))
gru_model.add(Dense(1,activation='sigmoid')) # Couche de sortie
# Compiler le modèle
gru_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy', 'precision'])

In [57]:

# Entrainer le modèle
history_gru = gru_model.fit(x_train, y_train, validation_data = (x_val, y_val), epochs = 20, callbacks= early_stopping)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.7552 - loss: 0.4863 - precision: 0.7953 - val_accuracy: 0.8683 - val_loss: 0.3041 - val_precision: 0.8565
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9456 - loss: 0.1439 - precision: 0.9487 - val_accuracy: 0.8460 - val_loss: 0.4013 - val_precision: 0.8093
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9858 - loss: 0.0438 - precision: 0.9878 - val_accuracy: 0.8527 - val_loss: 0.4374 - val_precision: 0.8584


In [58]:
# Accuracy et loss sur les données validation
print(f"Accuracy : {lstm_model.evaluate(x_val, y_val)[1]}  ---- Loss : {lstm_model.evaluate(x_val, y_val)[0]} -- Precision : {lstm_model.evaluate(x_val, y_val)[2]}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8772 - loss: 0.4228
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8772 - loss: 0.4228 
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8772 - loss: 0.4228


IndexError: list index out of range

###### <font color='#1dd1a1'> Test le modèle

In [59]:
#tweet non vectorisés
tweet_3 = tokenizer.sequences_to_texts(x_test[:10])
# tweet vectorisés
twt = x_test[:10]
twt_label = y_test[:10]


In [60]:
# Prédictions les labels
y_pred_10 = lstm_model.predict(twt)
# Arrondir les valeurs
y_pred_10 = np.round(y_pred_10)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step


In [61]:
# Afficher sentiments
for i in range(10):
  print(f"{tweet_3[i]} -----  {'Negative' if int(y_pred_10[i][0]) == 1 else 'Positive'} --- {'Negative' if twt_label[i] == 1 else 'Positive'}")

sma think hate white male hate racism sexism grow brain right winger p -----  Negative --- Negative
girl get fuck bondage leather girl -----  Negative --- Negative
baby melt hea smile baby daughter -----  Positive --- Positive
feeling mutual ableism islamophobia transphobia misogyny homophobia enablin bigot -----  Negative --- Negative
heading graduation mechtechplumbing futureplumber proudmama -----  Positive --- Positive
want obama catch mad cow disease caught w herford dy trial buried cow pasture -----  Negative --- Negative
allahsoil culture like horizon always view never reach -----  Negative --- Negative
unreal advisor refuse take payment say wait pcn customer service site working -----  Positive --- Positive
thankful good night sleep thankful positive -----  Positive --- Positive
mitchellrattle mitchell goodson -----  Positive --- Positive
